<a href="https://colab.research.google.com/github/RR-mao/MachineLearning/blob/main/%E3%80%8C0704_Colab_LINE_Bot_with_GEMINI_Tooluse_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 11.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://rework-bunt-shadily.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://rework-bunt-shadily.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch() #校長是最新的是因為使用了google search的工具
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology，簡稱明新科大）是一所位於台灣新竹縣新豐鄉的私立科技大學。學校創立於1966年3月，最初名為「明新工業專科學校」。歷經發展，於1997年改制為「明新技術學院」並附設專科部，最終在2002年9月奉教育部核准升格為「明新科技大學」。2018年12月，學校更名為「明新學校財團法人明新科技大學」。

學校名稱「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類的德性與情操，並以培育學生具備高尚品德、專業學問與優良技術為目標。明新科大佔地逾三十公頃，地處新竹縣新豐鄉，交通便利，校園環境優美。

明新科大現設有六個學院，包括半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院。學校的發展願景是成為「國際魅力產業科技大學」，並以培育「跨域整合、務實創新、全人學習」的專業人才為教育目標。

在教學特色方面，明新科大積極與產業合作，特別是在半導體、AI、元宇宙、風電綠能等前瞻產業領域。學校發展出MUST四大育才特色：多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation），旨在引導學生跨域學習，成為產業所需的技職人才。根據1111人力銀行統計，明新科大在半導體產業最愛聘用的畢業生中名列前茅，與頂尖大學並列，是唯一入榜的私立科大。學校也致力於推動智慧生活創新服務，建置「永續智慧商務」教學與實習場域，並導入生成式AI與AI專案應用，發展智慧零售、智慧金融、智慧製造與智慧商業等實驗室。此外，明新科大於2022年獲教育部核准通過「半導體科技博士學位學程」，這是該校成立以來的第一個博士班。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學的現任校長是**呂明峯**教授。他於2025年1月16日舉行布達暨交接典禮，並於2月1日正式上任，成為明新科技大學的第11任校長。

呂明峯校長在明新科大任教長達34年，曾擔任電子工程系主任、研發長、工學院院長、半導體學院院長、產學長等職務。他擁有豐富的業界經驗和學術背景，並在任內打造了全台首座半導體封裝測試類產線，推動成立半導體學院及半導體科技博士學位學程，使明新科大在半導體教育領域具有顯著的地位。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)